In [ ]:
ETL Project
E - Extract - get data from Excel and CSV files
T - Transform - using pandas fill null values, fix datatypes, remove duplicates
L - Load - load the clean data into SQL tables

In [10]:
#get the engine to read excel file
!pip install openpyxl

In [11]:
import pandas as pd
import mysql.connector

#-----Extract data into dataframes-------

#CSV files
orders_df = pd.read_csv("E:\\etl_project\\orders_data.csv")
#print(orders_df)
users_df = pd.read_csv("E:\\etl_project\\users_data.csv")
#print(users_df)
products_df = pd.read_csv("E:\\etl_project\\products_data.csv")

#print(products_df)

#EXCEL files
#engine - bridge which gets data from excel file
payments_df = pd.read_excel("E:\\etl_project\\payments_data.xlsx", engine = "openpyxl")
print(payments_df)
reviews_df = pd.read_excel("E:\\etl_project\\reviews_data.xlsx", engine = "openpyxl")
#print(reviews_df)

print("Extract Successful")
#orders_df.head()

    OrderID  PaymentMethod PaymentDate  Amount
0      1001  InvalidMethod         NaT     NaN
1      1002    Credit Card  2024-08-02  753.48
2      1003    Credit Card  2024-08-03  978.27
3      1004  InvalidMethod  2024-08-04  564.67
4      1005    Credit Card  2024-08-05  390.66
5      1006    Credit Card         NaT  815.67
6      1007  InvalidMethod  2024-08-07  343.75
7      1008    Credit Card  2024-08-08  495.07
8      1009    Credit Card  2024-08-09  170.61
9      1010  InvalidMethod  2024-08-10  122.82
10     1011    Credit Card         NaT     NaN
11     1012    Credit Card  2024-08-12  966.38
12     1013  InvalidMethod  2024-08-13  852.38
13     1014    Credit Card  2024-08-14  726.38
14     1015    Credit Card  2024-08-15  468.06
15     1016  InvalidMethod         NaT  255.96
16     1017    Credit Card  2024-08-17  240.79
17     1018    Credit Card  2024-08-18  325.22
18     1019  InvalidMethod  2024-08-19  594.30
19     1020    Credit Card  2024-08-20  743.14
20     1021  

In [12]:
#----------Transform data - cleaning

#fill missing values in each dataframe

#--------PRODUCTS TABLE
products_df = products_df.fillna({
    'Description': "No Description",
    'Category' : "Miscellaneous"
})

products_df['Price'] = pd.to_numeric(products_df['Price'], errors = 'coerce').fillna(0.0).round(2)
#to_numeric - changes datatype to number
# errors = 'coerce' - will show as NaN where value is not numeric
# NaN = Null, fillna - will replace the NaN with 0
#round - will set it as float value 
products_df['StockQuantity'] = pd.to_numeric(products_df['StockQuantity'], errors = 'coerce').fillna(0).round(0)

#print(products_df)

#-------ORDERS TABLE

orders_df = orders_df.fillna({
    'TotalAmount': 0.0
})

#orders_df['OrderDate'] = orders_df['OrderDate'].str.strip() # remove white spaces from string Invalid date)
orders_df['OrderDate'] = pd.to_datetime(orders_df['OrderDate'], errors = 'coerce')
orders_df['OrderDate'] = orders_df['OrderDate'].fillna(pd.Timestamp('1970-01-01'))
orders_df['OrderDate'] = orders_df['OrderDate'].dt.strftime('%Y-%m-%d')
#strftime() - converts date to given date format


# in dataframe add a new column which will have order ids's in range 1001 to total_records
# create original order id for mapping
orders_df["OldOrderID"] = range(1001, 1001 + len(orders_df))
#range - function which takes start value, end value and step value(1). fill the new column with values
#print(orders_df)

#--------USER TABLE

#function to generate random emailid
def generate_unique_email(index):
    return f'unknown{index}@example.com'
#index - index number assigned to each row in df    
# users_df['Email'] = users_df.apply(lambda row:row['Email'] if '@' in row['Email'] and '.' in row['Email'] else generate_unique_email(row.name) ,axis = 1)
users_df['Email'] = users_df.apply(
    lambda row: (
        row['Email']
        if pd.notna(row['Email'])
        and '@' in str(row['Email'])
        and '.' in str(row['Email'])
        else generate_unique_email(row.name)
    ),
    axis=1
)



#row.name - gets the row index number
#axis = 1 will check each row, axis = 0 - checks each column
#apply() - is a higher order function - takes another function as argument
#will apply the given function on each row
#print(users_df)

#------PAYMENTS TABLE
payments_df = payments_df.fillna(
    {
        'Amount':0.0
    }
)
payments_df['PaymentDate'] = pd.to_datetime(payments_df['PaymentDate'], errors = 'coerce', format = '%Y-%m-%d').fillna(pd.Timestamp('1970-01-01'))
payments_df['PaymentDate'] = payments_df['PaymentDate'].dt.strftime('%Y-%m-%d')

payments_df['PaymentMethod'] = payments_df['PaymentMethod'].replace('InvalidMethod','Unknown')
#print(payments_df)

#--------REVIEWS TABLE
reviews_df = reviews_df.fillna(
    {
        'ReviewText':'No Review Submitted!'
    }
)
reviews_df['Rating'] = reviews_df['Rating'].replace('InvalidRating',0)
reviews_df['Rating'] = pd.to_numeric(reviews_df['Rating'])
#print(reviews_df)

print("transform success")
    

transform success


C:\Users\balaj\AppData\Local\Temp\ipykernel_24104\3573223385.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders_df['OrderDate'] = pd.to_datetime(orders_df['OrderDate'], errors = 'coerce')
C:\Users\balaj\AppData\Local\Temp\ipykernel_24104\3573223385.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  reviews_df['Rating'] = reviews_df['Rating'].replace('InvalidRating',0)


In [13]:
# load data to database
#import mysql.connector

def conn():
    return mysql.connector.connect(
        host = "localhost",
        user = "root",
        password = "Surya@9373",
        database = "ecomm_data"
    )

con = conn()
cursor = con.cursor()

#delete previous records

#list of tables from which data to be cleared
tables_to_clear = ['Paymentss','Reviews','orderss','users','products']

#execute sql query in loop
for table in tables_to_clear:
    cursor.execute(f'delete from {table}')
con.commit() # save changes database

#adding records to product table
product_id_mapping = {} # empty dictionary to store the product_id's

for index, row in products_df.iterrows():
    # index - index number given to each row
    # row - hold one row at a time
    # iterrows() - iterate/loop through each row
    cursor.execute('''insert into products(product_name, p_description,price,stock_quantity,category) 
    values (%s,%s,%s,%s,%s)''',(row['ProductName'],row['Description'],row['Price'], row['StockQuantity'], row['Category']))
    product_id_mapping[index+1] = cursor.lastrowid
    # [index + 1] - key, lastrowid - value
con.commit()
print(product_id_mapping)

#adding records to user table
user_id_mapping = {}

for index, row in users_df.iterrows():
    cursor.execute('''insert into users(username, email, address, u_password) values (%s,%s,%s,%s)''',
                   (row['UserName'],row['Email'],row['Address'],row['Password']))
    user_id_mapping[index+1] = cursor.lastrowid
con.commit()
print(user_id_mapping)
#adding records to orders table

orders_df['UserID'] = orders_df['UserID'].map(user_id_mapping)
#map() will map the new values to each record
print(orders_df['UserID'])
orders_df = orders_df.dropna(subset = 'UserID')
# if any NULL userID delete the row

order_id_mapping = {}
for index, row in orders_df.iterrows():
    cursor.execute(''' insert into orders(userid, order_date, total_amt, email) values (%s,%s,%s,%s)''',
                   (row['UserID'], row['OrderDate'],row['TotalAmount'],row['Email']))
    order_id_mapping[row['OldOrderID']] = cursor.lastrowid
con.commit()
print(order_id_mapping)

#payments table 
payments_df['OrderID'] = payments_df['OrderID'].map(order_id_mapping)
print(payments_df['OrderID'])
payments_df = payments_df.dropna(subset=["OrderID"])

for index, row in payments_df.iterrows():
    cursor.execute('''insert into payments(orderid, payment_method,payment_date, amount) values (%s,%s,%s,%s)''',
                  (row['OrderID'], row['PaymentMethod'], row['PaymentDate'],row['Amount']))
con.commit()

# reviews table
reviews_df['UserID'] = reviews_df['UserID'].map(user_id_mapping)
reviews_df['ProductID'] = reviews_df['ProductID'].map(product_id_mapping)

for index, row in reviews_df.iterrows():
    product_id = product_id_mapping.get(index)
    user_id = user_id_mapping.get(index)
    cursor.execute('''insert into reviews(productid, userid, rating, review_text) values(%s,%s,%s,%s)''',
                  (int(row['ProductID']), int(row['UserID']), row['Rating'], row['ReviewText']))
con.commit()
print("load success")

{1: 361, 2: 362, 3: 363, 4: 364, 5: 365, 6: 366, 7: 367, 8: 368, 9: 369, 10: 370, 11: 371, 12: 372, 13: 373, 14: 374, 15: 375, 16: 376, 17: 377, 18: 378, 19: 379, 20: 380, 21: 381, 22: 382, 23: 383, 24: 384, 25: 385, 26: 386, 27: 387, 28: 388, 29: 389, 30: 390, 31: 391, 32: 392, 33: 393, 34: 394, 35: 395, 36: 396, 37: 397, 38: 398, 39: 399, 40: 400, 41: 401, 42: 402, 43: 403, 44: 404, 45: 405, 46: 406, 47: 407, 48: 408, 49: 409, 50: 410, 51: 411, 52: 412, 53: 413, 54: 414, 55: 415, 56: 416, 57: 417, 58: 418, 59: 419, 60: 420}
{1: 61, 2: 62, 3: 63, 4: 64, 5: 65, 6: 66, 7: 67, 8: 68, 9: 69, 10: 70}
0     68
1     64
2     61
3     68
4     64
5     66
6     68
7     64
8     63
9     69
10    63
11    69
12    62
13    62
14    62
15    66
16    63
17    69
18    64
19    61
20    64
21    61
22    65
23    64
24    68
25    68
26    67
27    63
28    61
29    61
30    63
31    66
32    67
33    66
34    66
35    66
36    63
37    66
38    68
39    62
40    65
41    61
42    61
43    65


ProgrammingError: 1146 (42S02): Table 'ecomm_data.orders' doesn't exist